In [1]:
import xarray as xr
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import os, re
import pandas as pd
os.chdir(r'J:\CMIP6_r1i1p1f1')

In [16]:
CMIP6_dir = r'J:\CMIP6_r1i1p1f1'
nc_list = [f[:-3] for f in os.listdir(CMIP6_dir) if f.endswith('.nc')]
csv_list = [f[:-4] for f in os.listdir('zonal_mean_df') if f.endswith('.csv')]
file_list =  [item for item in nc_list if item not in csv_list]

In [17]:
nc_list

['pr_Amon_ACCESS-CM2_ssp245_r1i1p1f1_gn_201501-210012',
 'pr_Amon_ACCESS-ESM1-5_historical_r1i1p1f1_gn_185001-201412',
 'pr_Amon_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_201501-210012',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185001-185012',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185101-185112',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185301-185312',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185401-185412',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185501-185512',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185701-185712',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_185801-185812',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186001-186012',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186101-186112',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186301-186312',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186401-186412',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186501-186512',
 'pr_Amon_AWI-CM-1-1-MR_historical_r1i1p1f1_gn_186901-186912',
 'p

In [18]:
len(file_list)

262

In [19]:
file_list

['pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210101-210112',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210201-210212',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210301-210312',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210401-210412',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210501-210512',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210601-210612',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210701-210712',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210801-210812',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210901-210912',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211001-211012',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211101-211112',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211201-211212',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211301-211312',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211401-211412',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211501-211512',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211601-211612',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211701-211712

In [20]:
['CanESM5', 'MIROC6', 'GISS-E2-1-G', 'IPSL-CM6A-LR', 'ACCESS-ESM1-5', 'ACCESS-CM2', 'CanESM5-1', 'EC-Earth3-Veg', 'NorESM2-LM']

for file_name in file_list:
    if int(file_name.split("_")[6][:4]) > 2100:
        print(file_name,' after 21st century')
        continue
    print(file_name)
    variable = file_name.partition('_')[0]
    GCM_name = file_name.split("_")[2]

    if GCM_name == 'IITM-ESM':
        continue
    # if GCM_name not in [ 'FGOALS-g3', 'MPI-ESM1-2-HR', 'EC-Earth3','BCC-CSM2-MR', 'MRI-ESM2-0', 'INM-CM5-0', 'INM-CM4-8']:
    #     continue
    
    # model_name = re.search(r'day_(.*?)_', file_name).group(1)
    index_df = pd.read_excel('lat_lon_index.xlsx',sheet_name=GCM_name)
    ds = xr.open_dataset(os.path.join(CMIP6_dir, file_name+'.nc'))


    lon_indices = index_df['lon_index'].tolist()
    lat_indices = index_df['lat_index'].tolist()
    # Extract the 'pr' variable from the dataset using the indices
    # Assuming 'lon_index' and 'lat_index' are valid indices for your dataset
    pr_data = ds[variable].isel(lon=lon_indices, lat=lat_indices)
    pr_df = pr_data.to_dataframe().reset_index()
    pr_df['lat'] = pr_df['lat'].round(6)
    pr_df['lon'] = pr_df['lon'].round(6)
    index_df['lat'] = index_df['lat'].round(6)
    index_df['lon'] = index_df['lon'].round(6)
    pr_df_merged = pr_df.merge(index_df, on=['lat','lon'],how='left')
    pr_df_merged.drop(['lat_index','lon_index', 'FID', 'Shape *'],axis=1,inplace=True)
    mean_pr_df = pr_df_merged.groupby(by=['time','abbre']).mean()
    # Pivot the DataFrame
    pr_df = mean_pr_df.reset_index().pivot(index='time', columns='abbre', values=variable)

    # Optional: Rename columns to make them more descriptive (if needed)
    pr_df.columns.name = None  # Remove the name of the columns index if undesired

    # Convert cftime.DatetimeNoLeap to pandas datetime
    pr_df.index = pr_df.index.map(lambda x: x.strftime('%Y-%m-%d') if hasattr(x, 'strftime') else x)
    pr_df.index = pd.to_datetime(pr_df.index)

    # Format time index as year-month
    pr_df.index = pr_df.index.to_period('D')
    # Display the resulting DataFrame
    pr_df.to_csv(os.path.join('zonal_mean_df', file_name+'.csv'))


pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210101-210112  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210201-210212  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210301-210312  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210401-210412  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210501-210512  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210601-210612  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210701-210712  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210801-210812  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_210901-210912  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211001-211012  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211101-211112  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211201-211212  after 21st century
pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_211301-211312  after 21st century
pr_Amon_EC-Earth3-Veg_ssp

In [24]:
file_list

['pr_Amon_ACCESS-CM2_ssp245_r1i1p1f1_gn_201501-210012',
 'pr_Amon_ACCESS-ESM1-5_ssp245_r1i1p1f1_gn_201501-210012',
 'pr_Amon_CanESM5-1_ssp245_r1i1p1f1_gn_201501-210012',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_201501-201512',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_201601-201612',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_201701-201712',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_201801-201812',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_201901-201912',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202001-202012',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202101-202112',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202201-202212',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202301-202312',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202401-202412',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202501-202512',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202601-202612',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202701-202712',
 'pr_Amon_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_202801-202812',
 'pr

In [13]:
file_name.split("_")[6][:4]

'2015'

In [5]:
file_name.split("_")[1][-3:]

NameError: name 'file_name' is not defined

In [5]:
index_df

NameError: name 'index_df' is not defined

# The follows is for get the index of each points in each basins and make a index table

In [8]:
basin_gdf = gpd.read_file(r'F:\geodata\river_runoff_obs\Tarim.shp')

In [9]:
model_list = []
for file_name in nc_list:
    model_name = file_name.split("_")[2]
    if model_name not in model_list:
        model_list.append(model_name)
    else:
        print(model_name)
        continue
    ds = xr.open_dataset(os.path.join(CMIP6_dir, file_name+'.nc'))
    lon_arr = ds.variables['lon'].values
    lat_arr = ds.variables['lat'].values
    lon_grid, lat_grid = np.meshgrid(lon_arr, lat_arr)

    # Create points and track indices
    points = []
    lon_indices = []
    lat_indices = []

    for lat_idx, lat in enumerate(lat_arr):
        for lon_idx, lon in enumerate(lon_arr):
            points.append(Point(lon, lat))  # Create Point object
            lon_indices.append(lon_idx)    # Track longitude index
            lat_indices.append(lat_idx)    # Track latitude index

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=points)
    gdf['lon'] = gdf.geometry.x  # Extract longitude (X-coordinate)
    gdf['lat'] = gdf.geometry.y  # Extract latitude (Y-coordinate)
    gdf['lon_index'] = lon_indices  # Add longitude indices
    gdf['lat_index'] = lat_indices  # Add latitude indices

    # Check if CRS is set
    if gdf.crs is None:
        # Set the CRS to WGS84 (latitude and longitude in degrees)
        gdf = gdf.set_crs("EPSG:4326")
    gdf = gdf.to_crs(basin_gdf.crs)
    # Keep only valid geometries
    gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty]

    gdf.to_file(f'{model_name}.shp', driver='ESRI Shapefile')
    del gdf

ACCESS-ESM1-5
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM-1-1-MR
AWI-CM

In [19]:
file_list

'pr_Amon_ACCESS-CM2_ssp245_r1i1p1f1_gn_201501-210012'